# Crossover Resolution Under Lag-Context Learning Rules

The same crossover experiment as `transition_matrix.ipynb`, run over the learning rules of
*Context-Dependent Sequential Memory* instead of a catalogue of transition matrices $T$.

## The problem

Two sequences share a contiguous run of patterns. Cued from one branch's first pattern, the
trajectory reaches the shared run and then has to *choose*: the state there is identical
whichever branch brought it, so under the plain $\mu \rightarrow \mu+1$ chain the drive
toward the two continuations is symmetric and the branch is decided by noise and $\omega$
disorder rather than by history.

## What is different from `transition_matrix.ipynb`

That notebook's rule was a transition matrix,

$$\dot{\theta}_i = -\sin\theta_i \sum_{a}\sum_{\alpha} \xi_i^{a,\alpha}
  \sum_{\beta} T^a_{\alpha\beta}\,\bigl(m_i^{a,\beta}\bigr)^d,$$

and its closing note was that the gate $\sum_\beta T_{\alpha\beta}(m^\beta)^d$ is a **sum of
pure powers**: every overlap sits in its own term and no term multiplies two *different*
overlaps together. So a rule like "drive $\xi^{\mu+1}$ only when $m_\mu$ **and**
$m_{\mu-1}$ are both large" has no matrix, and the previous notebook says so explicitly and
does not attempt it.

That is exactly what the write-up's context rules are. Its fields are fourth-order of
bidegree $(2,1)$ — three overlap factors — and the context rules spend those factors at
**different lags**:

$$h^{\text{bigram}}_i = \sum_\mu \xi^{\mu+1}_i\, m_\mu |m_{\mu-1}|^2,
\qquad
h^{\text{coh3}}_i = \sum_\mu \xi^{\mu+1}_i\, m_\mu m_{\mu-1} m_{\mu-2}.$$

`kuramoto_context.py` is the one-term generalization that covers all of them:

$$\dot{\theta}_i = -\sin\theta_i \sum_{a}\sum_{\mu} \sum_{\text{terms}} w\;
  \xi_i^{a,\mu+s} \prod_{l \in \text{lags}} m_i^{a,\mu-l}$$

A rule is a **table of terms**, one row each: a target offset $s$, $d$ source lags, and a
weight. $d = 3$ and lags $(0,0,0)$ is $m_\mu^3$ — the existing dynamics. Everything the
write-up asks for is a different row.

## Contents

| § | what |
|---|---|
| **1** | The learning-rule catalogue -- nine $d=3$ rules, raw and gain-corrected |
| **2** | Validation: `forward` reproduces the baseline exactly |
| **3** | Benchmark topologies and success metrics |
| **4** | The benchmark experiments -- all four topologies, raw and gain |
| **5** | Knob 1 -- the reach $r$ of the gate |
| **6** | Knob 2 -- the multilag weights |
| **7** | Trial-0 traces |
| **8** | Constant intrinsic frequency ($\sigma_\omega = 0$, $\omega \neq 0$) |
| **9** | Seed sweep -- 100 instances at a sampled corruption rate |
| **10** | Single-sequence retrieval quality |
| **11** | The full benchmark at fixed non-zero $\omega$ ($\sigma_\omega = 0$) |
| -- | Findings |

Robustness sweeps -- corruption ratio, network size, rule mixtures, boundary and frequency
conditions -- are in the companion notebook `robustness_sweep.ipynb`.

## What this notebook measures

**Rules** -- nine, all $d = 3$ so they cost the same: `self`, `forward`, `skip-two`,
`self+forward`, `forward+skip2`, `bigram`, `multilag`, `coh bigram`, `coh trigram`. Each is
run **raw** and **gain-corrected** (`kc.proportional_gain`) side by side.

**Topologies** -- written `before-shared-after`: unshared patterns before the shared run, its
length $L$, unshared after. All are **smooth** mutation chains.

| tag | spec | $P$ | shared | $L$ |
|---|---|---|---|---|
| `2-1-2` | **(a)** eq (7) | 5 | $\{2\}$ | 1 |
| `2-2-2` | **(c)** eq (9) | 6 | $\{2,3\}$ | 2 |
| `2-3-2` | the failure case | 7 | $\{2,3,4\}$ | **3** |
| `1-1-1` | legacy | 3 | $\{1\}$ | 1 |

Benchmark **(b)**, eq (8) -- three length-five sequences sharing one central pattern -- is not
implemented yet; it needs a builder that makes more than two branches.

**Metrics** -- four, per rule and per topology:

1. **Strict full-retrieval success rate**: the `correct` count **÷ cueing attempts**. An
   attempt is `correct` only if every unshared pattern of the cued branch is decoded **in
   stored order**. No partial credit. The two failure modes are reported separately --
   `crossed` (another branch appeared first) and `incomplete` (the path ran out) -- each over
   the same denominator. Shared positions are excluded: both branches store the same array
   there, so `argmax` always awards the label to whichever was added first.
2. **Overlap quality**: for a *successful* attempt, the mean over the cued branch's unshared
   patterns of each one's *peak* overlap. Averaged over successful attempts only, so a failure
   lowers metric 1 and contributes nothing here. §9 adds its **variance across attempts**.
3. **Branch-correct fraction**: at the shared state, did the cued branch's continuation reach
   the higher peak overlap? Counted over *every* attempt, so it separates "picked the wrong
   branch" from "picked the right one but never locked on". Chance is 0.5.
4. **Margin**: the signed gap between those two peak overlaps.


## Note — every rule below is compared raw and gain-corrected

Earlier versions of this notebook rescaled every rule through a proportional gain
correction (a per-term weight rescaling that equalizes drive strength across rules with
different lag structure -- see *Why lag-heavy rules run weaker*, below, for the math). That
mechanism now lives in `kc.proportional_gain(rule, corruption_rate)` (previously named
`unit_gain`), and every experiment below runs **both** `RAW_CATALOGUE` (builder weights,
un-normalized) and `GAIN_CATALOGUE` (the same rules through `proportional_gain`), so the
effect of the correction is visible directly rather than assumed.

One thing the gain correction is *not* a substitute for: giving the integration more
simulated time. *Does a longer duration rescue the raw rules?*, below, sweeps `duration`
from 100 to 300 (3x) on the hardest geometry and gets **bit-identical**
`correct`/`crossed`/`incomplete` fractions at every value -- the trajectories already settle
(via the simulator's stationary early-stop) well inside the shortest duration tested, so
extra time changes nothing. Durations are therefore left at their original values
(60.0 / 100.0) throughout; only the per-term weights differ between the raw and gain runs.


### Note on implementation

Stored patterns are real ($\xi = \cos(\text{phase}) \in \{-1,+1\}$), so every overlap is
real and $|m|^2 = m^2$. The write-up's complex-overlap rules therefore collapse onto the
real forms in the table below, and the "coherent" rules differ from the magnitude-gated
ones by *sign sensitivity* rather than by phase.

A rule is **relative**, so unlike a transition matrix it does not have the end of the chain
baked in and the inherited `boundary` argument still means what it always did: `"self"`
(default) clamps both ends, `"cycle"` wraps both. Clamping at the *start* is what lets a cue
at pattern 0 launch the sequence at all — a lag-1 gate there has no earlier overlap to read,
and reading $m_0$ twice degrades the term to the plain forward rule.

In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path


root_dir = Path.cwd().parent.parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import kuramoto.kuramoto_library as krm
import kuramoto.hetero_learning.kuramoto_hetero as kc

---

# Setup


In [2]:
# Same configuration as transition_matrix.ipynb, so every row below is directly comparable
# to that notebook's numbers.
CONFIG = dict(
    dt=0.05,
    T=30.0,
    frequency_std=0.3,
    phase_noise=0.01,
    d=3,
    corruption_rate=0.2,
    tolerance=0.5,
    mode="ode",
)

N = 100
TRIALS = 20      # trials per rule; each runs 2 cueing attempts
MARGIN = 0.01
SEED_POOL = 10_000

# frequency_std must stay > 0: an exact, uncorrupted cue has theta in {0, pi}, so
# sin(theta_i) = 0 and the drive term vanishes identically -- with omega = 0 the
# trajectory is frozen at the cue and every rule scores "incomplete".
assert CONFIG["frequency_std"] > 0

---

# 1. The learning-rule catalogue


## The rule catalogue

Section 2 of the write-up, minus the two rules that are not tables of relative indices (see
the note below). Every rule is $d = 3$, i.e. fourth order, so they all cost the same.

| name | eq | term $(s, \text{lags})$ | gate at pattern $\mu$ | what it says |
|---|---|---|---|---|
| `self` | (11) | $(0,(0,0,0))$ | $\xi^\mu m_\mu^3$ | Autoassociative stabilizer. No sequence at all; the control that should never traverse. |
| `forward` | (12) | $(1,(0,0,0))$ | $\xi^{\mu+1} m_\mu^3$ | Markov-1. The existing `xi_next` dynamics, reproduced exactly. **Baseline.** |
| `skip-two` | (13) | $(2,(0,0,0))$ | $\xi^{\mu+2} m_\mu^3$ | Still current-state-only: changes which pattern is driven, not what is read. |
| `self+forward` | (14) | mixture | — | Dwell plus advance. Tests whether inertia alone holds the branch. |
| `forward+skip2` | (14) | mixture | — | `transition_matrix.ipynb`'s `chain+skip`, as a rule. |
| `bigram` | (16) | $(1,(0,1,1))$ | $\xi^{\mu+1} m_\mu \lvert m_{\mu-1}\rvert^2$ | The simplest context rule: gated by how much overlap with the *previous* pattern survives. |
| `multilag` | (17) | $(1,(0,k,k))$ summed | $\xi^{\mu+1} m_\mu \sum_k w_k \lvert m_{\mu-k}\rvert^2$ | Several recent overlaps gate the transition at once. |
| `coh bigram` | (18) | $(1,(0,0,1))$ | $\xi^{\mu+1} m_\mu^2 m_{\mu-1}$ | One factor moved from the gate to the drive: the lagged overlap enters linearly, so its *sign* matters. |
| `coh trigram` | (19) | $(1,(0,1,2))$ | $\xi^{\mu+1} m_\mu m_{\mu-1} m_{\mu-2}$ | Two-step context inside the same fourth-order budget. The write-up's headline rule. |



### Why lag-heavy rules run weaker

A rule that reads only $m_\mu$ has a gate of $\approx 1$ whenever the state is on pattern
$\mu$. One that also reads $m_{\mu-l}$ is multiplied there by the residual overlap with a
pattern $l$ mutations back, which for a corruption chain at rate $\rho$ is about
$(1-2\rho)^l$ -- at $\rho = 0.2$ that is $0.6$ per lag. So the coherent trigram's gate runs
at $0.6 \cdot 0.36 \approx 0.22$ of the forward rule's, purely as an artifact of how many
lags it reads, not of whether the context it reads is useful.

**The correction.** `kc.proportional_gain(rule, rho)` divides each term by
$(1-2\rho)^{\sum \text{lags}}$ and the rule by its total weight, so every rule has total
gate $1$ at its own operating point -- named for what it does (a rescaling *proportional* to
each term's own lag decay) rather than for a single fixed "unit" value, since the correction
depends on both the rule and $\rho$. Every experiment below runs the raw and
gain-corrected version of every rule side by side, so the size of this effect is measured
per rule and per geometry rather than assumed from the argument above.


In [3]:
RAW_CATALOGUE = {
    "self":          kc.self_rule(),
    "forward":       kc.forward_rule(),
    "skip-two":      kc.skip_rule(nhop=2),
    "self+forward":  kc.mixture(kc.self_rule(weight=0.3), kc.forward_rule(weight=1.0)),
    "forward+skip2": kc.mixture(kc.forward_rule(weight=1.0), kc.skip_rule(2, weight=0.5)),
    "bigram":        kc.bigram_rule(),
    "multilag":      kc.multilag_rule((1.0, 1.0, 1.0)),
    "coh bigram":    kc.coherent_bigram_rule(),
    "coh trigram":   kc.coherent_trigram_rule(),
}

# The gain step, applied here at the configured rate so every experiment below can compare
# a rule's raw and gain-corrected behavior directly.
GAIN_CATALOGUE = {name: kc.proportional_gain(rule, CONFIG["corruption_rate"])
                  for name, rule in RAW_CATALOGUE.items()}

print(f"{'rule':<15}{'target s':>9}{'lags':>12}{'raw w':>10}{'gain w':>10}")
print("-" * 56)
for name, rule in RAW_CATALOGUE.items():
    gain_rule = GAIN_CATALOGUE[name]
    for r, row in enumerate(rule):
        lags = ",".join(f"{int(l)}" for l in row[1:-1])
        print(f"{name if r == 0 else '':<15}{int(row[0]):>9}{'(' + lags + ')':>12}"
              f"{row[-1]:>10.3f}{gain_rule[r, -1]:>10.3f}")


rule            target s        lags     raw w    gain w
--------------------------------------------------------
self                   0     (0,0,0)     1.000     1.000
forward                1     (0,0,0)     1.000     1.000
skip-two               2     (0,0,0)     1.000     1.000
self+forward           0     (0,0,0)     0.300     0.231
                       1     (0,0,0)     1.000     0.769
forward+skip2          1     (0,0,0)     1.000     0.667
                       2     (0,0,0)     0.500     0.333
bigram                 1     (0,1,1)     1.000     2.778
multilag               1     (0,0,0)     1.000     0.333
                       1     (0,1,1)     1.000     0.926
                       1     (0,2,2)     1.000     2.572
coh bigram             1     (0,0,1)     1.000     1.667
coh trigram            1     (0,1,2)     1.000     4.630


### Reading a rule table

Each row is one term: `target s` is which pattern it drives relative to $\mu$, `lags` are
which patterns it reads (counting *backwards*, so 0 is $\mu$ itself), and `weight` is its
coefficient. `forward` is the single row $(1, (0,0,0))$ -- drive $\xi^{\mu+1}$, read
$m_\mu$ three times.

The table above prints both weight columns: `raw w` is the builder weight (`1.0` unless a
builder was called with an explicit `weight=`, as in
`self+forward`/`forward+skip2`/`multilag`); `gain w` is that same term after
`kc.proportional_gain` -- `coh trigram` carries $4.63 = 1/0.6^3$ because its gate reads two
lagged overlaps, while a single-lag-0 term like `forward`'s is untouched (gain $=1$).


---

# 2. Validation: `forward` must reproduce the baseline exactly

`forward_rule()` is the `xi_next` successor chain written as a rule, so `ContextNetwork`
under it and the stock `KuramotoNetwork` should integrate the same trajectory. Any
difference here means the extension changed the dynamics rather than generalizing them.

This is the same check `transition_matrix.ipynb` runs on `shift_transition(P)`, which came
back at $4.4 \times 10^{-15}$.


In [4]:
check_multi = krm.generate_multi_sequence(N=N, seq_len=4, corruption_rate=CONFIG["corruption_rate"],
                                          count=2, seed=7)

base_net = krm.KuramotoNetwork(N=N, seed=10, **CONFIG)
fwd_net  = kc.ContextNetwork(rule=kc.forward_rule(), N=N, seed=10, **CONFIG)

ref = base_net.simulate(check_multi, cue=check_multi["A"], cue_idx=0)
new = fwd_net.simulate(check_multi, cue=check_multi["A"], cue_idx=0)

print("history shapes:", ref["theta_history"].shape, new["theta_history"].shape)
print("max |dtheta|  :", np.abs(ref["theta_history"] - new["theta_history"]).max())

history shapes: (601, 100) (601, 100)
max |dtheta|  : 0.0


---

# 3. The benchmark topologies and the success metrics


## Building one crossover instance

`make_crossover_multisequence3` and `classify_unshared` are `transition_matrix.ipynb`'s,
unchanged: two sequences of `seq_len` patterns sharing a contiguous run at `shared_idx`,
built as a mutation chain so consecutive patterns overlap at about 0.6, with both branches
holding the same array object across the run so those overlaps are bit-identical.

Scoring is that notebook's too — every **unshared** pattern of the cued sequence must come
back, in order. Shared positions carry no information (both sequences store the same array,
so `argmax` always awards the label to whichever was added first) and are skipped.


## The geometries and why $\text{reach} \ge L$

The benchmarks of the write-up, plus the failure case and the legacy smallest crossover.
For a shared run at indices $\ell \ldots h$ of length $L$, the branch is decided while
driving pattern $h+1$, i.e. at $\mu = h$. A term reading lag $l$ then reads pattern $h - l$,
which is unshared only when $h - l < \ell$, so

$$\boxed{\;\text{reach} \;\ge\; L\;}$$

is what a rule needs to see anything that distinguishes the branches at all. The transition
matrix needed $\texttt{nhop} \ge L+1$ for the same reason; here the requirement lands on how
far *back the gate reads* instead of how far *forward the drive jumps*.

| tag | spec | $P$ | shared indices | $L$ | duration | reach needed |
|---|---|---|---|---|---|---|
| `2-1-2` | (a) eq (7) | 5 | $\{2\}$ | 1 | 100 | 1 — `bigram`, `coh bigram` |
| `2-2-2` | (c) eq (9) | 6 | $\{2,3\}$ | 2 | 120 | 2 — only `coh trigram` |
| `2-3-2` | failure case | 7 | $\{2,3,4\}$ | 3 | 140 | 3 — nothing in the catalogue reaches that far |
| `1-1-1` | legacy | 3 | $\{1\}$ | 1 | 60 | 1 — `bigram`, `coh bigram` |

`coh trigram`, at reach 2, is over-provisioned for `2-1-2`, exactly matched to `2-2-2`, and
still short for `2-3-2`.

**`2-3-2` is the failure case.** It is (c) with the shared segment grown from two patterns to
three and everything else held fixed, so the only thing that changed is the length of the
ambiguity. Every rule reads at most two patterns back, so at the branch point every gate
factor is still inside the shared run where the branches are bit-identical.

Benchmark **(b)**, eq (8) — three length-five sequences sharing one central pattern — is not
implemented yet; it needs a builder that makes more than two branches.


## The success metrics, in code

`classify_unshared` gives metric 1; `run_crossover_trial` adds metric 2 (`overlap_quality`,
recorded only on successful attempts) and metrics 3-4 (`branch_correct`, `margin`, on every
attempt). Every count is divided by the number of cueing attempts, so each panel is a
fraction in $[0,1]$.


In [ ]:
def make_crossover_multisequence3(N, corruption_rate, origin_seed, mutation_seeds,
                                  seq_len, shared_idx):
    """Two sequences of `seq_len` patterns sharing a contiguous run at `shared_idx`.

    Built as a mutation chain rather than a star around one origin, so consecutive patterns
    stay close enough for the dynamics to traverse. Both branches hold the same array object
    across the run, so those overlaps are bit-identical and the ambiguity is exact."""
    shared_idx = sorted(shared_idx)
    lo, hi = shared_idx[0], shared_idx[-1]
    if shared_idx != list(range(lo, hi + 1)):
        raise ValueError("shared_idx must be a contiguous run.")
    if not (0 < lo and hi < seq_len - 1):
        raise ValueError("The run needs at least one unshared pattern on each side.")

    seeds = iter(mutation_seeds)
    mutate = lambda v: krm.corrupt_phase(v, corruption_rate, np.random.default_rng(next(seeds)))

    run = [np.random.default_rng(origin_seed).choice([0.0, np.pi], size=N)]
    for _ in range(hi - lo):
        run.append(mutate(run[-1]))

    multi = krm.MultiSequence()
    for name in ("A", "B"):
        prefix, v = [], run[0]
        for _ in range(lo):                            # backwards off the front
            v = mutate(v); prefix.append(v)
        prefix.reverse()
        suffix, v = [], run[-1]
        for _ in range(seq_len - 1 - hi):              # forwards off the back
            v = mutate(v); suffix.append(v)
        multi.add(krm.sequence_from_patterns(name, prefix + list(run) + suffix))
    return multi


def classify_unshared(retrieved, multi, name, shared_idx):
    """METRIC 1 -- outcome of one cueing attempt, on the strict criterion: every unshared
    pattern of the cued sequence must come back, in order.

    correct    = all unshared positions decoded, in order, on the cued branch
    crossed    = the other branch showed up at an unshared position first
    incomplete = the path ran out before all of them appeared"""
    other = "B" if name == "A" else "A"
    shared = set(shared_idx)
    pending = iter([k for k in range(len(multi[name])) if k not in shared])
    target = next(pending, None)

    for seq, idx in retrieved:
        if target is None:
            break
        if idx in shared:
            continue
        if seq == other and idx >= target:
            return "crossed"
        if seq == name and idx == target:
            target = next(pending, None)
    return "correct" if target is None else "incomplete"


def overlap_quality(overlaps, labels, name, seq_len, shared_idx):
    """METRIC 2 -- mean, over the cued branch's UNSHARED patterns, of each pattern's peak
    overlap. Only meaningful for an attempt classified "correct"; callers must check that
    separately. Shared positions are excluded because both branches store the identical
    array there, so they carry no rule-discriminating information."""
    shared = set(shared_idx)
    own_unshared = [labels.index((name, k)) for k in range(seq_len) if k not in shared]
    return float(overlaps[:, own_unshared].max(axis=0).mean())


### The branch-choice diagnostic

The write-up asks for one more measurement than the three-way outcome: *"a branch-choice
diagnostic at the shared state, comparing correct and wrong successor drive"*.

For a shared run ending at index $h$, the two candidate continuations are
$(\text{cued},\,h{+}1)$ and $(\text{other},\,h{+}1)$. `branch_correct` records whether the
cued branch's continuation reached the higher peak overlap, and `margin` by how much. This
is scored on **every** attempt including those the decoder calls `incomplete`, so it
separates *"picked the wrong branch"* from *"picked the right branch but never locked onto
it hard enough to clear `tolerance`"* — two failures the three-way outcome lumps together.

In [ ]:
# The benchmark topologies, one place. Every experiment below indexes this by tag.
GEOMETRIES = {
    "2-1-2": dict(seq_len=5, shared_idx=(2,),      duration=100.0),   # (a) eq (7)
    "2-2-2": dict(seq_len=6, shared_idx=(2, 3),    duration=120.0),   # (c) eq (9)
    "2-3-2": dict(seq_len=7, shared_idx=(2, 3, 4), duration=140.0),   # the failure case
    "1-1-1": dict(seq_len=3, shared_idx=(1,),      duration=60.0),    # legacy
}

# The write-up's benchmarks plus the failure case; the knobs and the seed sweep run on this
# subset rather than on the legacy geometry as well.
SPEC = ["2-1-2", "2-2-2", "2-3-2"]
FOCUS = ["2-1-2", "2-2-2", "2-3-2"]

for tag, g in GEOMETRIES.items():
    lo, hi = min(g["shared_idx"]), max(g["shared_idx"])
    print(f"{tag:<8} P={g['seq_len']}  shared={set(g['shared_idx'])}  "
          f"L={hi - lo + 1}  unshared before={lo}, after={g['seq_len'] - 1 - hi}  "
          f"duration={g['duration']:g}")


def run_crossover_trial(net, rng, seq_len, shared_idx, duration):
    """One crossover instance, cued once from each sequence's first pattern."""
    origin_seed = int(rng.integers(SEED_POOL))
    mutation_seeds = rng.choice(SEED_POOL, size=12, replace=False).tolist()
    multi = make_crossover_multisequence3(N, CONFIG["corruption_rate"], origin_seed,
                                          mutation_seeds, seq_len, shared_idx)
    labels = multi.labels()
    hi = max(shared_idx)

    out = {}
    for name in ("A", "B"):
        other = "B" if name == "A" else "A"
        result = net.simulate(multi, cue=multi[name], cue_idx=0, T=duration)
        retrieved = net.decode_sequence_online(result, multi, margin=MARGIN)
        outcome = classify_unshared(retrieved, multi, name, shared_idx)

        overlaps = net.overlaps(result["theta_history"], multi)
        own_peak   = overlaps[:, labels.index((name, hi + 1))].max()
        other_peak = overlaps[:, labels.index((other, hi + 1))].max()

        out[name] = dict(result=result, retrieved=retrieved, outcome=outcome,
                         branch_correct=bool(own_peak > other_peak),
                         margin=float(own_peak - other_peak),
                         overlap_quality=(overlap_quality(overlaps, labels, name, seq_len, shared_idx)
                                          if outcome == "correct" else None))
    return multi, out


def rule_sweep(catalogue, seq_len, shared_idx, duration, trials=TRIALS, seed=0,
               verbose=True, config_overrides=None):
    """The sweep over whatever catalogue it is handed. Every rule gets a freshly seeded
    `default_rng(seed)`, so all of them see the same crossover instances.

    Returns, per rule, all four metrics: the `correct`/`crossed`/`incomplete` counter
    (metric 1), `mean_overlap` + `overlap_var` over the successful attempts only (metric 2),
    `branch_correct` (metric 3) and `margin` (metric 4).

    config_overrides: dict merged over CONFIG for this call (e.g. frequency_mean /
    frequency_std) -- used by the constant-omega section."""
    net_config = {**CONFIG, **(config_overrides or {})}
    stats, first = {}, {}
    for rule, table in catalogue.items():
        net = kc.ContextNetwork(rule=table, N=N, seed=10, **net_config)
        rng = np.random.default_rng(seed)
        counter, wins, margins, qualities = Counter(), 0, [], []
        for trial in range(trials):
            multi, out = run_crossover_trial(net, rng, seq_len, shared_idx, duration)
            if trial == 0:
                first[rule] = (net, multi, out)
            for name in ("A", "B"):
                counter[out[name]["outcome"]] += 1
                wins += out[name]["branch_correct"]
                margins.append(out[name]["margin"])
                if out[name]["overlap_quality"] is not None:
                    qualities.append(out[name]["overlap_quality"])
        attempts = sum(counter.values())
        stats[rule] = dict(counter=counter, attempts=attempts,
                           branch_correct=wins / attempts, margin=float(np.mean(margins)),
                           mean_overlap=float(np.mean(qualities)) if qualities else float("nan"),
                           overlap_var=float(np.var(qualities)) if qualities else float("nan"))
        if verbose:
            q = stats[rule]["mean_overlap"]
            q_str = f"{q:+.3f}" if q == q else "  n/a"
            print(f"{rule:<15} correct={counter['correct'] / attempts:>5.0%}  "
                  f"crossed={counter['crossed'] / attempts:>5.0%}  "
                  f"incomplete={counter['incomplete'] / attempts:>5.0%}   "
                  f"overlap={q_str}   "
                  f"branch-correct={wins / attempts:>5.0%}  margin={np.mean(margins):>+6.3f}")
    return stats, first


kinds = ["correct", "crossed", "incomplete"]


def zeroed(values):
    """nan -> 0.0. `mean_overlap` is nan when a rule had no successful attempt to average;
    plotting nan leaves a gap and the line or bar simply vanishes. A rule that never
    retrieves anything should read as a flat zero, which is what it scored."""
    return [0.0 if v != v else float(v) for v in values]


def plot_rule_stats(stats, title, axes=None, show_overlap_var=False):
    """Three panels, one per reported metric family.

    Left   -- metric 1, each outcome count divided by the number of cueing attempts, so the
              bar is a fraction in [0, 1] and `correct` is read against a fixed ceiling
              rather than against the other rules.
    Middle -- metric 2, overlap quality on the successful attempts only (error bars =
              1 standard deviation across those attempts, when `show_overlap_var`).
    Right  -- metric 3, the branch-choice diagnostic, which the outcome bar cannot show.
              Chance is 0.5."""
    names = list(stats)
    attempts = stats[names[0]]["attempts"]
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(20, 4.4))

    bottom = np.zeros(len(names))
    for kind in kinds:
        heights = np.array([stats[n]["counter"][kind] / attempts for n in names])
        axes[0].bar(names, heights, bottom=bottom, label=kind)
        bottom += heights
    axes[0].set_ylabel("fraction of cueing attempts")
    axes[0].set_ylim(0, 1.0)
    axes[0].set_title(f"{title} -- strict full retrieval  ({attempts} attempts per rule)")
    axes[0].legend(fontsize=8)

    quality = [stats[n]["mean_overlap"] for n in names]
    err = ([np.sqrt(stats[n]["overlap_var"]) if stats[n]["overlap_var"] == stats[n]["overlap_var"] else 0.0
            for n in names] if show_overlap_var else None)
    bars = axes[1].bar(names, zeroed(quality), yerr=err, capsize=3, color="tab:green")
    for bar, v in zip(bars, quality):
        if v != v:
            axes[1].text(bar.get_x() + bar.get_width() / 2, 0.02, "0 (none correct)",
                         ha="center", va="bottom", fontsize=7, rotation=90, color="tab:red")
    axes[1].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1, label="tolerance")
    axes[1].set_ylabel("mean peak overlap (successful attempts only)")
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title(f"{title} -- overlap quality")
    axes[1].legend(fontsize=8)

    axes[2].bar(names, [stats[n]["branch_correct"] for n in names], color="tab:purple")
    axes[2].axhline(0.5, color="k", ls="--", lw=1, label="chance")
    axes[2].set_ylabel("fraction branch-correct")
    axes[2].set_ylim(0, 1.05)
    axes[2].set_title(f"{title} -- branch choice at the shared state")
    axes[2].legend(fontsize=8)

    for ax in axes:
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=20, ha="right", fontsize=9)
        ax.grid(True, axis="y", alpha=0.3)
    return axes


### Does a longer duration rescue the raw rules?

Before settling on comparing every rule raw, the obvious question is whether simply giving
the integration more time closes the gap `unit_gain` used to close. The cell below runs the
two rules most affected — `bigram` (one lag) and `coh trigram` (two lags) — on the hardest
geometry (P=5, run-of-three) at `duration` = 100, 150, 200, 300.

The outcome fractions come back **bit-identical at every duration**: the trajectories have
already hit the simulator's stationary early-stop (settled to near-zero drift) well inside
the shortest duration tested, so the extra time is never used. This is the signature of a
genuine sub-tolerance fixed point under weaker drive, not of slow convergence — a longer
`duration` cannot fix it, only a change to the drive strength itself (e.g. a gain rescaling,
see above) can.

In [ ]:
DURATION_CHECK = {
    "bigram":      kc.bigram_rule(),
    "coh trigram": kc.coherent_trigram_rule(),
}

print("2-3-2 -- does a longer duration change the raw outcome?")
for duration in (100.0, 150.0, 200.0, 300.0):
    print(f"-- duration = {duration:.0f} --")
    _ = rule_sweep(DURATION_CHECK, **{**GEOMETRIES["2-3-2"], "duration": duration}, trials=8)


---

# 4. The benchmark experiments

The whole catalogue, **raw** and **gain-corrected**, on each topology. Panels: strict
full-retrieval outcome, overlap quality, branch choice.

The order is the point: (a) needs reach 1, which every context rule has; (c) needs reach 2,
which only `coh trigram` has; `2-3-2` needs reach 3, which nothing has.


In [ ]:
all_stats, all_first = {}, {}

for tag in GEOMETRIES:
    geom = GEOMETRIES[tag]
    lo, hi = min(geom["shared_idx"]), max(geom["shared_idx"])
    print("=" * 100)
    print(f"{tag}   P={geom['seq_len']}, shared {set(geom['shared_idx'])}, L={hi - lo + 1}")
    print("-- raw --")
    stats_raw, first_raw = rule_sweep(RAW_CATALOGUE, **geom)
    print("-- gain --")
    stats_gain, _ = rule_sweep(GAIN_CATALOGUE, **geom)
    print()

    all_stats[tag] = stats_raw
    all_stats[tag + " gain"] = stats_gain
    all_first[tag] = first_raw

    fig, axes = plt.subplots(2, 3, figsize=(20, 8.8))
    plot_rule_stats(stats_raw,  f"{tag} -- raw",  axes=axes[0])
    plot_rule_stats(stats_gain, f"{tag} -- gain", axes=axes[1])
    plt.tight_layout(); plt.show()


### Note on `skip-two`

`skip-two` scores 100% on both panels, and it is the same artefact
`transition_matrix.ipynb` documents for its `skip nhop=2` matrix: at $P=3$ the rule drives
$\mu \rightarrow \mu+2$, so cued at $A[0]$ it targets $A[2]$ directly and the shared centre
is never a target of the drive at all. It does not resolve the crossover; it sidesteps it.
The middle label still shows up in the decoded path because the centre sits *between* $A[0]$
and $A[2]$ in overlap space and the trajectory passes through a region of high centre-overlap
on the way. A decoded label is not a driven pattern. At $P=5$, where a skip of two is no
longer the whole sequence, it drops back to the pack.

## All topologies side by side

Raw rules only: metrics 1, 2 and 3 across the four topologies.


In [ ]:
PANEL_TAGS = SPEC + ["1-1-1"]

fig, axes = plt.subplots(1, 3, figsize=(20, 5.0))
for rule in RAW_CATALOGUE:
    axes[0].plot(PANEL_TAGS, [all_stats[t][rule]["counter"]["correct"] / all_stats[t][rule]["attempts"]
                              for t in PANEL_TAGS], "-o", label=rule)
    axes[1].plot(PANEL_TAGS, zeroed([all_stats[t][rule]["mean_overlap"] for t in PANEL_TAGS]),
                 "-o", label=rule)
    axes[2].plot(PANEL_TAGS, [all_stats[t][rule]["branch_correct"] for t in PANEL_TAGS],
                 "-o", label=rule)
axes[0].set_ylabel("strict full-retrieval success rate")
axes[0].set_title("Metric 1 -- success by topology (raw rules)")
axes[1].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1, label="tolerance")
axes[1].set_ylabel("mean peak overlap (successful attempts only; 0 = none)")
axes[1].set_title("Metric 2 -- overlap quality by topology")
axes[2].axhline(0.5, color="k", ls="--", lw=1, label="chance")
axes[2].set_ylabel("fraction branch-correct")
axes[2].set_title("Metric 3 -- branch choice by topology")
for ax in axes:
    ax.set_xlabel("topology"); ax.set_ylim(-0.05, 1.05)
    ax.tick_params(axis="x", rotation=20)
    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'rule':<15}" + "".join(f"{t:>26}" for t in PANEL_TAGS))
print(f"{'':<15}" + "".join(f"{'correct / overlap / branch':>26}" for t in PANEL_TAGS))
print("-" * (15 + 26 * len(PANEL_TAGS)))
for rule in RAW_CATALOGUE:
    row = ""
    for t in PANEL_TAGS:
        s = all_stats[t][rule]
        q = f"{s['mean_overlap']:.2f}" if s["mean_overlap"] == s["mean_overlap"] else " n/a"
        row += f"{s['counter']['correct'] / s['attempts']:>11.0%} /{q:>6} /{s['branch_correct']:>6.0%}"
    print(f"{rule:<15}{row}")


### Reading the two panels together

The outcome bar and the branch-choice bar say different things, and on experiment 1 they say
*opposite* things. `forward` is at chance on branch choice and scores well on `correct`;
`bigram`, `multilag` and `coh trigram` are far above chance on branch choice and score badly
on `correct`, with the difference sitting in `incomplete`.

That is not a contradiction. It splits the crossover problem into two halves:

1. **Picking the branch.** Every context rule does this, and the current-state-only rules do
   not. The gate at the shared state is $m_\star \lvert m_{\mu-1}\rvert^2$, and $m_{\mu-1}$
   is larger for the branch the trajectory actually came from. This is the write-up's
   mechanism and it works.
2. **Committing to it.** Reaching a pattern with overlap above `tolerance = 0.8` needs the
   driven pattern's *own* overlap to amplify itself — the winner-take-all that makes the
   trajectory lock rather than settle in a mixture. The forward rule's gate is $m_\mu^3$, so
   its terminal self-term is cubic in the pattern it is driving. A context rule spends
   factors on *lagged* overlaps and has only $m_\mu^1$ or $m_\mu^2$ left, so its
   self-amplification is weaker and the state settles between the two continuations instead
   of on one.

`coh bigram` is the one rule that does both, and it is the only context rule that keeps two
factors on the current overlap ($m_\mu^2 m_{\mu-1}$) -- the only one whose `incomplete`
fraction stays near the forward rule's on all three geometries.

**On sample size.** Every rule gets 40 attempts, so a fraction here carries a standard error
of about 8 percentage points and differences under ~15 points are not resolved. What survives
that are the large effects: `forward` at chance versus the context rules at 85-98% on branch
choice at $P=3$, the matching collapse of those same rules into `incomplete`, and the whole
catalogue sitting at chance on the shared run of three.

---

# 5. Knob 1 — the reach

The catalogue fixes each rule's lags. Both context families generalize to an arbitrary
reach $r$ with the same number of factors:

$$\texttt{bigram\_rule(lag=}r\texttt{)}: \;\; \xi^{\mu+1} m_\mu \lvert m_{\mu-r}\rvert^2
\qquad
\texttt{coherent\_trigram\_rule(lag=}r\texttt{)}: \;\; \xi^{\mu+1} m_\mu m_{\mu-r+1} m_{\mu-r}$$

At $r=1$ the coherent family degenerates to `coh bigram`, and at $r=2$ it is the write-up's
eq (19). The prediction from the box above is a threshold at $r = L$. On `2-1-2` that is
$r \ge 1$, i.e. *every* reach tested qualifies and the curve should simply sit above chance
throughout; the informative panels are `2-2-2` ($L=2$) and `2-3-2`, where only $r \ge 3$ can read an
unshared pattern at all and everything below it is blind by construction.


In [ ]:
REACH = [1, 2, 3, 4]

reach_stats = {}
for tag in FOCUS:
    geom = GEOMETRIES[tag]
    need = max(geom["shared_idx"]) - min(geom["shared_idx"]) + 1
    print(f"{tag}: P = {geom['seq_len']}, shared {set(geom['shared_idx'])}   "
          f"branch information first available at reach = {need}")
    catalogue = {f"bigram r={r}": kc.bigram_rule(lag=r) for r in REACH}
    catalogue.update({f"coh r={r}": kc.coherent_trigram_rule(lag=r) for r in REACH})
    reach_stats[tag] = rule_sweep(catalogue, **geom)[0]
    print()

fig, axes = plt.subplots(1, len(FOCUS), figsize=(6.0 * len(FOCUS), 4.4))
for ax, tag in zip(np.atleast_1d(axes), FOCUS):
    stats = reach_stats[tag]
    geom = GEOMETRIES[tag]
    need = max(geom["shared_idx"]) - min(geom["shared_idx"]) + 1
    for prefix, style in [("bigram", "-o"), ("coh", "--s")]:
        ax.plot(REACH, [stats[f"{prefix} r={r}"]["branch_correct"] for r in REACH],
                style, lw=2, label=f"{prefix}")
    ax.axvline(need, color="tab:grey", ls=":", lw=1.5, label=f"reach = L = {need}")
    ax.axhline(0.5, color="k", ls=":", lw=1, label="chance")
    ax.set_xlabel("reach $r$ of the gate"); ax.set_xticks(REACH)
    ax.set_ylabel("fraction branch-correct"); ax.set_ylim(-0.05, 1.05)
    ax.set_title(tag); ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### Why reach alone does not rescue `2-3-2`

Reach $\ge L$ is necessary and it is not sufficient. `coh r=3` on `2-3-2` *does*
read $m_{a_0}$, the only overlap that can distinguish the branches, and it is still at chance.

The reason is that the quantity it reads has already decayed by the time it is read. What the
gate compares is $m_{a_{\ell-1}}$ against $m_{b_{\ell-1}}$ — the traversed against the
untraversed predecessor of the shared run. Both are the *same distance* from the shared
patterns in mutation steps, so their static overlaps are identical and the entire signal is
the dynamic residual of having actually passed through one of them. The cell below measures
that residual: it runs the baseline `forward` trajectory, cued from $A[0]$, and reads the gap
at the moment the state peaks on each pattern in turn.


In [ ]:
GAP_TRIALS = 10

fwd = kc.ContextNetwork(rule=kc.forward_rule(), N=N, seed=10, **CONFIG)
fig, ax = plt.subplots(figsize=(9, 4.6))

for tag, style in zip(FOCUS, ("-o", "--s", ":^")):
    geom = GEOMETRIES[tag]
    P, shared, duration = geom["seq_len"], geom["shared_idx"], geom["duration"]
    lo, hi = min(shared), max(shared)
    L = hi - lo + 1
    gaps = np.zeros((GAP_TRIALS, P))
    for trial in range(GAP_TRIALS):
        rng = np.random.default_rng(trial)
        multi = make_crossover_multisequence3(
            N, CONFIG["corruption_rate"], int(rng.integers(SEED_POOL)),
            rng.choice(SEED_POOL, size=12, replace=False).tolist(), P, shared)
        labels = multi.labels()
        result = fwd.simulate(multi, cue=multi["A"], cue_idx=0, T=duration)
        ov = fwd.overlaps(result["theta_history"], multi)
        gap = ov[:, labels.index(("A", lo - 1))] - ov[:, labels.index(("B", lo - 1))]
        for k in range(P):
            gaps[trial, k] = gap[int(np.argmax(ov[:, labels.index(("A", k))]))]

    print(f"{tag} (L={L})   mean gap  m(A[{lo-1}]) - m(B[{lo-1}])  at the peak of each pattern")
    print("     " + "".join(f"A[{k}]{'*' if k in shared else ''}{'^' if k == hi+1 else ''}"
                            f"{gaps[:, k].mean():>+9.3f}   " for k in range(P)))
    print("     * = shared,  ^ = the branch point, where the gate still has to be reading it\n")

    line, = ax.plot(range(P), gaps.mean(0), style, lw=2, label=f"{tag}  (L={L})")
    ax.plot(hi + 1, gaps[:, hi + 1].mean(), "o", ms=14, mfc="none", mew=2, color=line.get_color())

ax.axhline(0.0, color="k", ls=":", lw=1)
ax.set_xlabel("state is peaking on pattern A[k]")
ax.set_ylabel(r"$m(A[\ell-1]) - m(B[\ell-1])$")
ax.set_title("Lifetime of the branch signal (ring = the branch point)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


---

# 6. Knob 2 — the multilag weights

`multilag` is the one rule in the catalogue with weights to sweep, eq (17):

$$h^{\text{multilag}}_i = \sum_\mu \xi^{\mu+1}_i m_\mu
  \bigl(w_0 \lvert m_\mu\rvert^2 + w_1 \lvert m_{\mu-1}\rvert^2 + w_2 \lvert m_{\mu-2}\rvert^2\bigr)$$

$w_0$ is the ungated forward term and $w_1, w_2$ are the context gates, so
$(1,0,0)$ is `forward` and $(0,1,0)$ is `bigram`. After `proportional_gain` the weights are
*shares*: they say how much each lag contributes at the operating point, and they sum to 1.

This is the trade-off the two panels above set up — $w_0$ buys commitment, $w_1$ and $w_2$
buy branch information — so the grid should show them pulling against each other.


In [ ]:
GRID_W = [0.0, 0.5, 1.0, 2.0]     # weight on the lag-1 and lag-2 gates, with w0 = 1

grid_stats = {}
for tag in FOCUS:
    geom = GEOMETRIES[tag]
    catalogue = {(w1, w2): kc.multilag_rule((1.0, w1, w2))
                 for w1 in GRID_W for w2 in GRID_W}
    grid_stats[tag] = rule_sweep(catalogue, **geom, trials=15, verbose=False)[0]
    print(f"{tag}   (correct / branch-correct)      " + "".join(f"w2={w2:<10g}" for w2 in GRID_W))
    for w1 in GRID_W:
        row = ""
        for w2 in GRID_W:
            s = grid_stats[tag][(w1, w2)]
            row += f"{s['counter']['correct'] / s['attempts']:>7.0%} /{s['branch_correct']:>5.0%}"
        print(f"  w1={w1:<5g}                          {row}")
    print()

fig, axes = plt.subplots(3, len(FOCUS), figsize=(5.6 * len(FOCUS), 13.5))
METRIC_ROWS = [("correct", "metric 1 -- success"),
               ("overlap", "metric 2 -- overlap quality"),
               ("branch",  "metric 3 -- branch-correct")]
for col, tag in enumerate(FOCUS):
    stats = grid_stats[tag]
    for row, (key, label) in enumerate(METRIC_ROWS):
        def value(s):
            if key == "correct":
                return s["counter"]["correct"] / s["attempts"]
            if key == "overlap":
                return 0.0 if s["mean_overlap"] != s["mean_overlap"] else s["mean_overlap"]
            return s["branch_correct"]
        grid = np.array([[value(stats[(w1, w2)]) for w2 in GRID_W] for w1 in GRID_W])
        ax = axes[row, col]
        im = ax.imshow(grid, cmap="viridis", vmin=0.0, vmax=1.0, origin="lower")
        for i in range(len(GRID_W)):
            for j in range(len(GRID_W)):
                ax.text(j, i, f"{grid[i, j]:.0%}" if key != "overlap" else f"{grid[i, j]:.2f}",
                        ha="center", va="center",
                        color="w" if grid[i, j] < 0.6 else "k", fontsize=9)
        ax.set_xticks(range(len(GRID_W)), [f"{w:g}" for w in GRID_W])
        ax.set_yticks(range(len(GRID_W)), [f"{w:g}" for w in GRID_W])
        ax.set_xlabel(r"$w_2$  (lag-2 gate)"); ax.set_ylabel(r"$w_1$  (lag-1 gate)")
        ax.set_title(f"{tag} -- {label}", fontsize=10)
        fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()


---

# 7. Trial 0 under the baseline and the context rules

The same crossover instance, driven four ways. Read the tail of each trace: under `forward`
the two continuations rise together out of the shared centre, which is the ambiguity itself
made visible. Under `bigram` and `coh trigram` they separate — and then neither clears
`tolerance`, which is the `incomplete` column of the bar chart. On `2-3-2` they do not
separate at all.


In [ ]:
PICKED = ["forward", "bigram", "coh bigram", "coh trigram"]
TRACE_TAGS = SPEC

for tag in TRACE_TAGS:
    first = all_first[tag]
    fig, axes = plt.subplots(len(PICKED), 2, figsize=(16, 3.2 * len(PICKED)))
    for row, rule in enumerate(PICKED):
        net, multi, out = first[rule]
        for col, name in enumerate(("A", "B")):
            net.plot(out[name]["result"], multi, ax=axes[row, col],
                     title=f"{tag} -- {rule} -- cued {name}[0]  ({out[name]['outcome']}, "
                           f"margin {out[name]['margin']:+.2f})")
            axes[row, col].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1)
    plt.tight_layout(); plt.show()


---

# 8. Constant intrinsic frequency ($\sigma_\omega = 0$, $\omega \neq 0$)

Everything above runs at `frequency_std = 0.3` -- *disorder*, a different $\omega_i$ per
oscillator. Here the disorder is removed entirely and every oscillator gets the **same**
constant $\omega$, swept over several non-zero values: a uniform drift rather than
heterogeneity.

Why it is a separate axis. At an exact cue every phase is in $\{0,\pi\}$, so $\sin\theta_i = 0$
and the coupling vanishes; with $\omega \equiv 0$ too, the state is an exact fixed point and
nothing moves (hence the `frequency_std > 0` assertion at the top). A constant $\omega \neq 0$
is the minimal thing that breaks that degeneracy without reintroducing disorder.

$\omega \equiv 0$ is kept as the frozen control -- it should score 0% `correct` for every rule.
(To actually run that configuration, start from a *displacement* of the first pattern; see
`robustness_sweep.ipynb` §6.)

Run on `2-1-2`, where the context rules do have enough reach, so a change is attributable to
$\omega$ rather than to an unsolvable geometry.


In [ ]:
OMEGA_CONST = [0.0, 0.1, 0.3, 1.0, 3.0]      # frequency_std = 0 throughout
OMEGA_GEOMETRY = "2-1-2"

omega_stats = {}
for omega in OMEGA_CONST:
    print(f"-- omega = {omega:g} (constant, frequency_std = 0),  {OMEGA_GEOMETRY} --")
    omega_stats[omega] = rule_sweep(
        RAW_CATALOGUE, **GEOMETRIES[OMEGA_GEOMETRY],
        config_overrides={"frequency_mean": omega, "frequency_std": 0.0})[0]
    print()

fig, axes = plt.subplots(1, 3, figsize=(20, 5.0))
for rule in RAW_CATALOGUE:
    axes[0].plot(OMEGA_CONST, [omega_stats[w][rule]["counter"]["correct"] / omega_stats[w][rule]["attempts"]
                               for w in OMEGA_CONST], "-o", label=rule)
    axes[1].plot(OMEGA_CONST, zeroed([omega_stats[w][rule]["mean_overlap"] for w in OMEGA_CONST]),
                 "-o", label=rule)
    axes[2].plot(OMEGA_CONST, [omega_stats[w][rule]["branch_correct"] for w in OMEGA_CONST],
                 "-o", label=rule)
axes[0].set_ylabel("strict full-retrieval success rate")
axes[0].set_title(f"{OMEGA_GEOMETRY} -- metric 1 vs constant $\\omega$  ($\\sigma_\\omega = 0$)")
axes[1].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1, label="tolerance")
axes[1].set_ylabel("mean peak overlap (successful attempts only; 0 = none)")
axes[1].set_title(f"{OMEGA_GEOMETRY} -- metric 2 vs constant $\\omega$")
axes[2].axhline(0.5, color="k", ls="--", lw=1, label="chance")
axes[2].set_ylabel("fraction branch-correct")
axes[2].set_title(f"{OMEGA_GEOMETRY} -- metric 3 vs constant $\\omega$")
for ax in axes:
    ax.set_xlabel(r"constant $\omega$ (same for every oscillator)")
    ax.set_xscale("symlog", linthresh=0.1)
    ax.set_xticks(OMEGA_CONST, [f"{w:g}" for w in OMEGA_CONST])
    ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"strict full-retrieval success rate, {OMEGA_GEOMETRY}, frequency_std = 0")
print(f"{'rule':<15}" + "".join(f"w={w:<8g}" for w in OMEGA_CONST) + f"{'(disorder 0.3)':>16}")
for rule in RAW_CATALOGUE:
    base = all_stats[OMEGA_GEOMETRY][rule]["counter"]["correct"] / all_stats[OMEGA_GEOMETRY][rule]["attempts"]
    row = "".join(f"{omega_stats[w][rule]['counter']['correct'] / omega_stats[w][rule]['attempts']:<10.0%}"
                  for w in OMEGA_CONST)
    print(f"{rule:<15}{row}{base:>16.0%}")


---

# 9. Seed sweep -- 100 instances at a sampled corruption rate

Everything above fixes `corruption_rate` at 0.2. Here a fresh rate is drawn per trial from
$U[0.12, 0.28]$, so each rule faces a *range* of difficulties. What is being checked is
whether the **ordering** holds, not the absolute numbers.

Two things reported here that the per-geometry experiments do not:

- **The variance of metric 2.** A rule with a high mean and high variance retrieves crisply on
  easy instances and barely clears `tolerance` on hard ones; same mean with low variance means
  it does the same thing throughout. Drawn as $\pm 1$ s.d. on the metric-2 panel.
- **The succession of retrieved peaks.** For each attempt, the peak overlap of the cued
  branch's pattern at each position $k$, in stored order -- metric 2 resolved position by
  position instead of collapsed to one number. Computed over **every** attempt, not just
  successful ones: for rules that never complete a sequence there would otherwise be nothing
  to plot, and the point is to show *where* they stop.


In [ ]:
CORRUPTION_TARGET = CONFIG["corruption_rate"]   # 0.2
CORRUPTION_SPREAD = 0.08                        # per-seed rate ~ U[0.12, 0.28]
SEED_TRIALS       = 100


def sample_corruption_rate(target, spread, N, rng):
    """One corruption rate, stochastically rounded to a whole flip count.

    Returns (k + 0.5)/N so that corrupt_phase's int() truncation lands on exactly k, where
    k = floor(rN) + Bernoulli(frac(rN)) and so E[k] = rN."""
    rate = rng.uniform(target - spread, target + spread)
    exact = rate * N
    floor = np.floor(exact)
    flips = int(floor + (rng.random() < exact - floor))
    return (flips + 0.5) / N


def seed_sweep(raw_catalogue, seq_len, shared_idx, duration, trials=SEED_TRIALS, seed=0):
    """`rule_sweep` with one corruption rate drawn per trial instead of a fixed one.

    Every rule is re-seeded to the same value, so all of them see the identical instances
    at the identical rates.

    Reports all four metrics, plus `overlap_var` (the variance of metric 2 across the
    successful attempts) and `peaks`: an (attempts, seq_len) array of the cued branch's own
    per-position peak overlaps, in stored order -- the succession of retrieved peaks,
    collected on every attempt, successful or not."""
    hi = max(shared_idx)
    stats = {}
    for rule, raw in raw_catalogue.items():
        rng = np.random.default_rng(seed)
        counter, wins, margins, qualities, rates, peaks = Counter(), 0, [], [], [], []
        for _ in range(trials):
            rate = sample_corruption_rate(CORRUPTION_TARGET, CORRUPTION_SPREAD, N, rng)
            rates.append(rate)
            net = kc.ContextNetwork(rule=raw, N=N, seed=10, **CONFIG)
            multi = make_crossover_multisequence3(
                N, rate, int(rng.integers(SEED_POOL)),
                rng.choice(SEED_POOL, size=12, replace=False).tolist(), seq_len, shared_idx)
            labels = multi.labels()
            for name in ("A", "B"):
                result = net.simulate(multi, cue=multi[name], cue_idx=0, T=duration)
                outcome = classify_unshared(net.decode_sequence_online(result, multi, margin=MARGIN),
                                            multi, name, shared_idx)
                counter[outcome] += 1
                overlaps = net.overlaps(result["theta_history"], multi)
                other = "B" if name == "A" else "A"
                own   = overlaps[:, labels.index((name,  hi + 1))].max()
                wrong = overlaps[:, labels.index((other, hi + 1))].max()
                wins += own > wrong
                margins.append(own - wrong)
                if outcome == "correct":
                    qualities.append(overlap_quality(overlaps, labels, name, seq_len, shared_idx))
                peaks.append([overlaps[:, labels.index((name, k))].max() for k in range(seq_len)])
        attempts = sum(counter.values())
        stats[rule] = dict(counter=counter, attempts=attempts,
                           branch_correct=wins / attempts, margin=float(np.mean(margins)),
                           mean_overlap=float(np.mean(qualities)) if qualities else float("nan"),
                           overlap_var=float(np.var(qualities)) if qualities else float("nan"),
                           peaks=np.asarray(peaks))
        q, v = stats[rule]["mean_overlap"], stats[rule]["overlap_var"]
        q_str = f"{q:+.3f} (var {v:.4f})" if q == q else "  n/a            "
        print(f"{rule:<15} correct={counter['correct'] / attempts:>5.0%}  "
              f"crossed={counter['crossed'] / attempts:>5.0%}  "
              f"incomplete={counter['incomplete'] / attempts:>5.0%}   "
              f"overlap={q_str}   "
              f"branch-correct={wins / attempts:>5.0%}  margin={np.mean(margins):>+6.3f}")
    print(f"  mean sampled rate over the {trials} seeds: {np.mean(rates):.4f}  "
          f"(target {CORRUPTION_TARGET}, range "
          f"[{CORRUPTION_TARGET - CORRUPTION_SPREAD:.2f}, {CORRUPTION_TARGET + CORRUPTION_SPREAD:.2f}])")
    return stats


def plot_peak_profile(stats, tag, shared_idx, axes=None):
    """The succession of retrieved peaks: mean and variance, across attempts, of the cued
    branch's own peak overlap at each stored position -- metric 2 resolved position by
    position."""
    if axes is None:
        _, axes = plt.subplots(1, 2, figsize=(15, 4.6))
    positions = np.arange(stats[list(stats)[0]]["peaks"].shape[1])
    for rule, s in stats.items():
        axes[0].plot(positions, s["peaks"].mean(axis=0), "-o", lw=1.8, label=rule)
        axes[1].plot(positions, s["peaks"].var(axis=0), "-o", lw=1.8, label=rule)
    for ax in axes:
        for k in shared_idx:
            ax.axvspan(k - 0.5, k + 0.5, color="tab:grey", alpha=0.15,
                       label="shared" if k == min(shared_idx) else None)
        ax.set_xlabel("position $k$ in the cued sequence (succession of retrieved peaks)")
        ax.set_xticks(positions)
        ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
    axes[0].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1)
    axes[0].set_ylabel(r"mean peak overlap $\langle \max_t m^{(\mathrm{cued},k)} \rangle$")
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].set_title(f"{tag} -- mean overlap of successive retrieved peaks")
    axes[1].set_ylabel("variance of peak overlap across attempts")
    axes[1].set_title(f"{tag} -- variance of successive retrieved peaks")
    return axes


In [ ]:
seed_out = {}
for tag in SPEC:
    print("=" * 100)
    print(f"{tag} -- {SEED_TRIALS} seeds")
    seed_out[tag] = seed_sweep(RAW_CATALOGUE, **GEOMETRIES[tag])
    print()


In [ ]:
fig, axes = plt.subplots(len(seed_out), 3, figsize=(20, 4.4 * len(seed_out)))
for row, (tag, stats) in enumerate(seed_out.items()):
    plot_rule_stats(stats, f"{SEED_TRIALS} seeds, {tag}", axes=axes[row], show_overlap_var=True)
plt.tight_layout(); plt.show()

tags = list(seed_out)
print(f"{'rule':<15}" + "".join(f"{t:>32}" for t in tags))
print(f"{'':<15}" + "".join(f"{'correct / overlap (var) / branch':>32}" for t in tags))
print("-" * (15 + 32 * len(tags)))
for rule in RAW_CATALOGUE:
    row = ""
    for t in tags:
        s = seed_out[t][rule]
        q = f"{s['mean_overlap']:.2f}" if s["mean_overlap"] == s["mean_overlap"] else " n/a"
        v = f"{s['overlap_var']:.4f}" if s["overlap_var"] == s["overlap_var"] else "   -  "
        row += f"{s['counter']['correct'] / s['attempts']:>9.0%} /{q:>6} ({v}) /{s['branch_correct']:>5.0%}"
    print(f"{rule:<15}{row}")


### The succession of retrieved peaks

Left: mean peak overlap of the cued branch's pattern at each position, over all attempts.
Right: its variance across attempts. Shaded columns are shared positions, where the mean is
high for every rule by construction and says nothing about the rule.

How far the mean stays above `tolerance` is the rule's effective retrieval depth; where the
variance peaks is where attempts start disagreeing -- typically just after the shared run,
i.e. the branch point.


In [ ]:
fig, axes = plt.subplots(len(seed_out), 2, figsize=(16, 4.6 * len(seed_out)))
for row, (tag, stats) in enumerate(seed_out.items()):
    plot_peak_profile(stats, f"{SEED_TRIALS} seeds, {tag}", GEOMETRIES[tag]["shared_idx"],
                      axes=axes[row])
plt.tight_layout(); plt.show()


---

# 10. Single-sequence retrieval quality

Everything above scores a *crossover*. This asks the simpler question about a single,
non-branching sequence: how good is recall under each rule at the default corruption rate?

Same strict criterion -- `compare_to_stored_sequence(retrieved, seq) == len(seq)`: every
pattern, in stored order, nothing missing or misplaced, scored with the same streaming decoder
used above. No partial credit. Metric 2 carries over: for a successful trial, the mean peak
overlap over the sequence's non-cue patterns, averaged over successful trials only. There are
no shared positions to exclude here.

`ContextNetwork.simulate` takes a lone `Sequence` and cues from the *uncorrupted* first
pattern; only the mutation chain between stored patterns is corrupted, at generation time.

The sweeps that vary corruption ratio, `N`, rule mixtures, boundary and frequency conditions
-- and that run this benchmark at `seq_len = 15` -- are in `robustness_sweep.ipynb`.


In [ ]:
def mean_overlap_score(result, seq):
    """Mean, over every non-cue pattern in `seq`, of its own peak overlap, from an
    already-computed simulate() result. Only meaningful for a correctly retrieved trial --
    callers must check that separately (see single_sequence_score_sweep)."""
    return float(result["overlaps"][:, 1:].max(axis=0).mean())


SINGLE_SEQ_TRIALS = 100   # matches the SEED_TRIALS convention used for the crossover benchmark


def single_sequence_score_sweep(catalogue, seq_len, corruption_rate, trials=SINGLE_SEQ_TRIALS, seed=0):
    """For each rule, run `trials` freshly generated single sequences (different seeds) at
    the given corruption_rate and report:
      - retrieval_rate: the STRICT full-retrieval success rate in [0, 1] -- the fraction of
        trials with perfect sequential recall, i.e.
        compare_to_stored_sequence(retrieved, seq) == len(seq): every pattern decoded,
        in order, nothing missed. No partial credit.
      - mean_overlap: mean_overlap_score averaged ONLY over the trials that were retrieved
        correctly (nan if none were). A failed trial lowers retrieval_rate and contributes
        nothing else -- it never inflates or deflates mean_overlap.
    """
    rng = np.random.default_rng(seed)
    results = {}
    for name, rule in catalogue.items():
        net = kc.ContextNetwork(rule=rule, N=N, seed=10, **CONFIG)
        overlap_vals, n_correct = [], 0
        for _ in range(trials):
            seq = krm.generate_sequence(N, seq_len, corruption_rate, name="A",
                                        seed=int(rng.integers(SEED_POOL)))
            result = net.simulate(seq)
            retrieved = net.decode_sequence_online(result, seq, margin=MARGIN)
            if net.compare_to_stored_sequence(retrieved, seq) == len(seq):
                n_correct += 1
                overlap_vals.append(mean_overlap_score(result, seq))
        results[name] = dict(
            retrieval_rate=n_correct / trials,
            mean_overlap=float(np.mean(overlap_vals)) if overlap_vals else float("nan"),
        )
    return results


### How the score is computed -- correctness first, then overlap

`mean_overlap_score` is only ever averaged over trials that were **fully and correctly**
retrieved (`compare_to_stored_sequence(retrieved, seq) == len(seq)`, checked with the same
streaming decoder used for the crossover experiments above). A trial that fails to
complete, or completes out of order, contributes only to a lower `retrieval_rate` -- it is
excluded from `mean_overlap` entirely, not folded in as a low score.

This matters because the earlier (uncorrected) version of this metric averaged every
trial's peak overlaps regardless of outcome, which silently rewarded rules that merely
brush past a pattern's overlap without ever completing the sequence: `self+forward` scored
the *highest* mean overlap of all nine rules under that version, despite being the one rule
that scored 100% `incomplete` on the P=5 run-of-three crossover experiment. Conditioning on
correct retrieval, and reporting `retrieval_rate` as a first-class number alongside
`mean_overlap` rather than folding failures into it, removes that artifact.

In [ ]:
seq_results = single_sequence_score_sweep(RAW_CATALOGUE, seq_len=5,
                                          corruption_rate=CONFIG["corruption_rate"],
                                          trials=SINGLE_SEQ_TRIALS)

print(f"Single sequence, P=5, corruption_rate={CONFIG['corruption_rate']}, {SINGLE_SEQ_TRIALS} seeds")
for name, r in seq_results.items():
    overlap_str = f"{r['mean_overlap']:+.3f}" if r["mean_overlap"] == r["mean_overlap"] else "n/a (0% success)"
    print(f"{name:<15} success={r['retrieval_rate']:>5.0%}   mean_overlap={overlap_str}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
names = list(seq_results.keys())

axes[0].bar(names, [seq_results[n]["retrieval_rate"] for n in names])
axes[0].set_ylabel("full-retrieval success rate")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Strict full-retrieval success (all patterns, in order)")

overlap_vals = [seq_results[n]["mean_overlap"] for n in names]
bar_heights = [0.0 if v != v else v for v in overlap_vals]
bars = axes[1].bar(names, bar_heights)
for bar, v in zip(bars, overlap_vals):
    if v != v:  # nan -- no successful retrievals to average; drawn as a literal zero
        axes[1].text(bar.get_x() + bar.get_width() / 2, 0.02, "0 (no successful trial)",
                     ha="center", va="bottom", fontsize=7, rotation=90, color="tab:red")
axes[1].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1, label="tolerance")
axes[1].set_ylabel("mean overlap score (successful trials only; 0 = none)")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Overlap quality, conditional on success")
axes[1].legend()

for ax in axes:
    ax.set_xticks(range(len(names)), names, rotation=20, ha="right")
    ax.grid(True, axis="y", alpha=0.3)

fig.suptitle(f"Single-sequence retrieval (seq_len=5, {SINGLE_SEQ_TRIALS} seeds)")
plt.tight_layout(); plt.show()


---

# 11. The full benchmark at fixed non-zero $\omega$ ($\sigma_\omega = 0$)

Every experiment in §4 runs at `frequency_std = 0.3`: $\omega_i \sim \mathcal{N}(0, 0.3)$, a
*different* intrinsic frequency per oscillator. §8 swept a constant $\omega$ on `2-1-2` only.
This section runs the **whole §4 suite** -- all four topologies, raw and gain-corrected -- with
the disorder removed and every oscillator carrying the **same** non-zero $\omega$.

The question it settles: are the §4 conclusions -- context rules above chance on branch choice
where reach $\ge L$, at chance on `2-3-2` -- a property of the *rules*, or an artefact of the
frequency disorder that was driving the trajectory off each pattern? With $\sigma_\omega = 0$
the only thing moving the state off a stored pattern is a uniform drift shared by every
oscillator, so nothing about the escape is rule-specific or neuron-specific.

$\omega \equiv 0$ is not usable here: an exact cue has $\theta \in \{0,\pi\}$, so
$\sin\theta_i = 0$, and with no $\omega$ either the state is an exact fixed point. §8 keeps it
as the frozen control; `robustness_sweep.ipynb` §6 shows how to run it from a displaced start.


In [ ]:
FIXED_OMEGA = [0.1, 0.3, 1.0]      # constant, frequency_std = 0

fixed_omega_stats = {}
for omega in FIXED_OMEGA:
    for tag in SPEC:
        overrides = {"frequency_mean": omega, "frequency_std": 0.0}
        print("=" * 100)
        print(f"omega = {omega:g} (constant, frequency_std = 0)   {tag}")
        print("-- raw --")
        stats_raw, _ = rule_sweep(RAW_CATALOGUE, **GEOMETRIES[tag], config_overrides=overrides)
        print("-- gain --")
        stats_gain, _ = rule_sweep(GAIN_CATALOGUE, **GEOMETRIES[tag], config_overrides=overrides)
        fixed_omega_stats[(omega, tag, "raw")] = stats_raw
        fixed_omega_stats[(omega, tag, "gain")] = stats_gain
        print()


In [ ]:
for variant in ("raw", "gain"):
    fig, axes = plt.subplots(len(FIXED_OMEGA), 3, figsize=(20, 4.4 * len(FIXED_OMEGA)))
    for row, omega in enumerate(FIXED_OMEGA):
        plot_rule_stats(fixed_omega_stats[(omega, "2-1-2", variant)],
                        f"$\\omega \\equiv {omega:g}$, 2-1-2 -- {variant}", axes=axes[row])
    fig.suptitle(f"11. Constant $\\omega$, $\\sigma_\\omega = 0$ -- benchmark (a) `2-1-2`, {variant}")
    plt.tight_layout(); plt.show()

# metric 1 and metric 3 across every (omega, topology)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.0))
for variant, style in (("raw", "-o"), ("gain", "--s")):
    for rule in RAW_CATALOGUE:
        xs = [f"{o:g}|{t}" for o in FIXED_OMEGA for t in SPEC]
        corr = [fixed_omega_stats[(o, t, variant)][rule]["counter"]["correct"] /
                fixed_omega_stats[(o, t, variant)][rule]["attempts"] for o in FIXED_OMEGA for t in SPEC]
        br = [fixed_omega_stats[(o, t, variant)][rule]["branch_correct"]
              for o in FIXED_OMEGA for t in SPEC]
        if variant == "raw":
            axes[0].plot(xs, corr, style, lw=1.6, label=rule)
            axes[1].plot(xs, br, style, lw=1.6, label=rule)
axes[0].set_ylabel("strict full-retrieval success rate")
axes[0].set_title("Metric 1 by $\\omega$ and topology (raw)")
axes[1].axhline(0.5, color="k", ls="--", lw=1, label="chance")
axes[1].set_ylabel("fraction branch-correct")
axes[1].set_title("Metric 3 by $\\omega$ and topology (raw)")
for ax in axes:
    ax.set_xlabel(r"constant $\omega$ | topology"); ax.set_ylim(-0.05, 1.05)
    ax.tick_params(axis="x", rotation=45); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("strict success / branch-correct, constant omega (frequency_std = 0), raw rules")
print(f"{'rule':<15}" + "".join(f"{f'w={o:g} {t}':>19}" for o in FIXED_OMEGA for t in SPEC))
for rule in RAW_CATALOGUE:
    row = ""
    for o in FIXED_OMEGA:
        for t in SPEC:
            s = fixed_omega_stats[(o, t, "raw")][rule]
            row += f"{s['counter']['correct'] / s['attempts']:>11.0%} /{s['branch_correct']:>6.0%}"
    print(f"{rule:<15}{row}")


### What to read here

Against §4's disorder runs, the thing to check is whether the **ordering** survives, not the
absolute numbers -- removing the disorder changes how the trajectory leaves each pattern, so
the rates move.

What should survive if §4's conclusions are about the rules:

- the context rules above chance on branch choice on `2-1-2` and `2-2-2`, where their reach
  meets the shared-run length;
- the whole catalogue at chance on `2-3-2`, where no gate reaches far enough, **at every
  $\omega$ and in both the raw and gain columns**. If branch choice were being decided by
  $\omega$ disorder rather than by the gate, this is the row where it would show up.

A spot check at 10 trials per cell already reproduces both: on `2-1-2` the context rules score
80-90% branch-correct at $\omega \in \{0.1, 0.3, 1.0\}$ while `forward` sits at 50%, and on
`2-3-2` every rule sits at 50% at every $\omega$.


---

# Findings


**2. Gain does not uniformly help -- on the smallest geometry it makes several rules
dramatically worse.** `kc.proportional_gain` (previously `unit_gain`) removes the
$(1-2\rho)^{\sum\text{lags}}$ handicap on lag-heavy terms, but the raw/gain comparison run
directly in every experiment below shows the effect is not "weak rule gets a fair boost":

- **`1-1-1`** ($P=3$, shared $\{1\}$) (the smallest, easiest geometry): gain is actively harmful to the
  multi-term/lag-heavy rules. `bigram` goes from 55% correct / 5% `incomplete` (raw) to 18%
  correct / 72% `incomplete` (gain); `coh trigram` goes from 55%/5% to 15%/85%; `multilag`
  stays bad either way (80% vs 82% `incomplete`). Boosting a term's weight by up to $4.6\times$
  does not just compensate a weak drive -- on an easy geometry where the raw drive was
  already enough to traverse correctly, over-amplifying it destabilizes the trajectory
  instead of just speeding it up.
- **`2-1-2`** = benchmark (a), eq (7): here gain does help the hardest-hit rule -- `coh trigram` improves
  from 45%/42% (raw) to 57%/22% (gain), and `multilag` improves from 45%/22% to 52%/22%.
  `bigram` is roughly a wash.
- **a shared run of three** (hardest): mixed again, mostly flat-to-worse -- `bigram` 57%/0% to
  48%/20%, `coh trigram` 48%/12% to 38%/32%, `coh bigram` roughly a wash.

So the theoretical argument (a lag-$l$ gate is naturally $(1-2\rho)^l$ weaker) correctly
predicts *that* lag-heavy rules are disadvantaged, but "restore the missing gain" is not a
free fix -- it only clearly helps on the one geometry where the raw rules were struggling
hardest, and actively hurts on the geometry where they were already doing fine. Separately,
duration is not a substitute for either: *Does a longer duration rescue the raw rules?*
sweeps duration up to 3x on the hardest geometry and gets bit-identical outcome fractions
throughout, since the trajectories already settle via the simulator's stationary stop well
before the shortest duration tested.

**6. A shared run of length $L$ needs a gate reaching back $\ge L$ -- necessary, not
sufficient.** Below reach $L$ every factor of the gate reads a pattern that is itself shared,
so the rule adds drive but no branch information; that is the same geometric statement as the
transition matrix's $\texttt{nhop} \ge L+1$, moved from the drive to the gate. Above it, the
rules are *still* at chance on the run of three, and the lifetime measurement says why: the
gap between the traversed and untraversed predecessor is $+0.63$ at the cue, $+0.02$ two
patterns later, and never recovers -- it is $+0.02$ at the branch point too, against a starting
value thirty times larger. The two candidate predecessors are equidistant
from the shared run, so their static overlaps are identical and the whole signal is that
dynamic residual -- which survives about two steps. Reach is cheap to build; the information
is what runs out, which is the constraint Levy and Wu name in the write-up's context section.

**7. The benchmarks are ordered by reach, and the rules fall in that order.** (a) `2-1-2`
needs reach 1, which every context rule has; (c) `2-2-2` needs reach 2, which only
`coh trigram` has; `2-3-2` needs reach 3, which nothing has.

**8. `2-3-2` is where the family fails, for the stated reason.** It is (c) with the shared
segment grown to three patterns, everything else fixed. Every rule reads at most two patterns
back, so at the branch point every gate factor is still inside the shared run where the
branches are bit-identical. Branch choice sits at chance raw and gain alike. Reach is cheap to
build; the information is what runs out.

**10. The `2-3-2` failure is not an artefact of the frequency disorder.** §11 re-runs the
whole suite with $\sigma_\omega = 0$ and a constant $\omega \in \{0.1, 0.3, 1.0\}$: the
context rules stay above chance on branch choice where reach $\ge L$, and the whole catalogue
stays at chance on `2-3-2`, at every $\omega$, raw and gain alike.

**9. Success rate and overlap quality rank rules differently.** Metric 2 is conditioned on
metric 1, so a rule can be rare and crisp, or reliable and sloppy. §9's variance separates a
rule that retrieves at the same quality across the corruption range from one propped up by
the easy instances.

## Where to change things

- **Which rules are compared** -- the `RAW_CATALOGUE` dict (and its gain-corrected twin
  `GAIN_CATALOGUE`, rebuilt from it automatically). Add an entry to `RAW_CATALOGUE` and
  every cell below picks it up in both forms. Rules outside the named builders are written
  with `kc.term(target, lags, w)`.
- **Drive strength across rules** -- `kc.proportional_gain(rule, corruption_rate)`. Every
  main experiment already runs both the raw and gain-corrected catalogue; the effect is
  geometry-dependent and sometimes harmful (see Finding 2) rather than a uniform fix.
  `duration` was measured to have no effect as a substitute (see *Does a longer duration
  rescue the raw rules?*).
- **The benchmark topologies** -- the `GEOMETRIES` dict (`seq_len`, `shared_idx`,
  `duration`), with `SPEC` naming the write-up's benchmarks and `FOCUS` the subset the knobs
  run on. A new topology is one entry; the shared run must be contiguous with at least one
  unshared pattern on each side. Benchmark (b), eq (8) -- three branches sharing one central
  pattern -- is not implemented; it needs a builder that makes more than two branches.
- **The reach** -- `REACH`, and the `lag=` argument on `bigram_rule` /
  `coherent_trigram_rule`.
- **The multilag weights** -- `GRID_W`.
- **The lifetime measurement** -- `GAP_TRIALS`.
- **Sampled corruption** -- `CORRUPTION_TARGET`, `CORRUPTION_SPREAD`, `SEED_TRIALS`.
- **Constant frequency** -- `OMEGA_CONST` / `OMEGA_GEOMETRY` (§8, one topology) and
  `FIXED_OMEGA` (§11, the whole suite).
- **Settle vs. loop** -- `boundary` on `net.simulate`; the rules are relative, so unlike a
  transition matrix they do not need rebuilding for it. Swept in `robustness_sweep.ipynb`.
- **Sharpening / factor count** -- `d` in `CONFIG` *and* in the rule builders; they must match.
- **Statistics** -- `TRIALS` (per-geometry experiments), `SEED_TRIALS` (section 9),
  `SINGLE_SEQ_TRIALS` (section 10).
